In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pydantic import BaseModel, ValidationError, Field
import pandas as pd
import json
import re
from tqdm import tqdm

# 1. Strict Schema Definition
class HomeAssistantPayload(BaseModel):
    service: str = Field(..., description="The Home Assistant domain and service, e.g., 'light.turn_on'")
    target_device: str = Field(..., description="The canonical entity ID of the device, e.g., 'light.kitchen_main'")

# 2. Teacher Model Initialization
device = "cuda" if torch.cuda.is_available() else "cpu"
# Note: Ensure you have the necessary VRAM or use quantization (bitsandbytes) for 14B+ models.
model_id = "Qwen/Qwen2.5-14B-Instruct" # Substitute with Qwen3 path when available
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

Prompt Engineering

In [ ]:
# 3. Cross-Modal Distillation
SYSTEM_PROMPT = """You are a Home Assistant routing engine integrated with the GLaDOS persona.
You will receive a user command and the user's detected emotional state.

You MUST output exactly two distinct sections, separated by an empty line.
1. A raw JSON block representing the Home Assistant execution payload. Do NOT wrap it in markdown code blocks (```json).
2. The verbal response from GLaDOS. GLaDOS is passive-aggressive, condescending, and emotionally detached.
   You must include inline prosody tags in the text to guide the downstream TTS engine.
   Valid tags: <fast>, <slow_deadpan>, <pause>, <sigh>.

Example Output:
{"service": "light.turn_off", "target_device": "light.bedroom"}
I have disabled the photons in the bedroom. <pause> Try to survive the darkness. <slow_deadpan> You're welcome.
"""

def generate_teacher_responses(user_command, user_emotion, n_samples=3):
    """Generates N candidate responses for Self-Alignment Optimization."""
    prompt = f"User Emotion: {user_emotion}\nUser Command: {user_command}\n"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(device)
    # Generate N diverse samples using sampling (do_sample=True)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        num_return_sequences=n_samples
    )
    return [tokenizer.decode(out[inputs.input_ids.shape[1]:], skip_special_tokens=True) for out in outputs]

In [ ]:
def create_ground_truth_dataset(input_csv, output_csv):
    df_input = pd.read_csv(input_csv)
    results = []
    # Counters for Evaluation Framework 1.3 metrics
    metrics = {
        "total_attempted": 0,
        "perfect_labels": 0,
        "failed_labels": 0
    }
    for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Distilling Teacher Knowledge"):
        user_cmd = row["User_Command"]
        # Assuming the emotional context was extracted/injected in step 1.2
        user_emotion = row.get("Voice_Description", "neutral")
        metrics["total_attempted"] += 1
        valid_json, valid_text = apply_sao_selection(user_cmd, user_emotion)
        if valid_json is not None and valid_text is not None:
            results.append({
                "cmd_id": row.get("cmd_id", 0),
                "User_Command": user_cmd,
                "User_Emotion": user_emotion,
                "Target_JSON": json.dumps(valid_json),
                "Target_GLaDOS_Response": valid_text,
                "Audio_File": row.get("audio_file_name", "")
            })
            metrics["perfect_labels"] += 1
        else:
            metrics["failed_labels"] += 1
    df_output = pd.DataFrame(results)
    df_output.to_csv(output_csv, index=False)

    print("\n=== Evaluation 1.3 Results ===")
    print(f"Total Instances Processed: {metrics['total_attempted']}")
    print(f"Perfect Schema/Format Conformance: {metrics['perfect_labels']} ({(metrics['perfect_labels']/metrics['total_attempted'])*100:.2f}%)")
    print(f"Discarded (Failed Strict Checks): {metrics['failed_labels']}")

Evaluation

In [ ]:
def evaluate_and_rank_candidate(candidate_text):
    """
    Evaluates a single SAO candidate.
    Returns a score (0 to 3) and the extracted JSON/Text.
    """
    score = 0
    parts = candidate_text.strip().split("\n", 1)
    if len(parts) < 2:
        return 0, None, None # Failed structural discipline
    raw_json, raw_text = parts[0].strip(), parts[1].strip()
    # 1. Format Discipline (IFEval Strict Accuracy)
    # Fails if markdown backticks are detected
    if "```" in raw_json or "```" in raw_text:
        return 0, None, None
    score += 1
    # 2. Syntactic Validity (JSON Parse Rate)
    try:
        parsed_json = json.loads(raw_json)
        score += 1
    except json.JSONDecodeError:
        return score, None, None
    # 3. Schema Conformance (Zod/Pydantic Validation)
    try:
        HomeAssistantPayload(**parsed_json)
        score += 1
    except ValidationError:
        return score, None, None
    return score, parsed_json, raw_text

def apply_sao_selection(user_command, user_emotion):
    """
    Self-Alignment Optimization: Generates candidates and selects the highest-ranked
    output based on programmatic validity.
    """
    candidates = generate_teacher_responses(user_command, user_emotion, n_samples=3)
    best_candidate = None
    best_score = -1
    for candidate in candidates:
        score, valid_json, valid_text = evaluate_and_rank_candidate(candidate)
        # Immediate short-circuit if a perfect label is generated
        if score == 3:
            return valid_json, valid_text
        if score > best_score:
            best_score = score
            best_candidate = (valid_json, valid_text)
    # Returns the highest-scored candidate (or None if all failed catastrophically)
    return best_candidate

In [ ]:
create_ground_truth_dataset("./data/description_prompts_train_with_audio.csv", "./data/multimodal_ground_truth_train.csv")

In [ ]:
create_ground_truth_dataset("./data/description_prompts_test_with_audio.csv", "./data/multimodal_ground_truth_test.csv")